# Generative AI - Assignment 1

**Name:** Yuvraj Singh
**Student ID:** 27PGAI0086

This notebook has two parts:

- **Part 1** - Topic detection, summarization and entity extraction on the BBC News dataset (first 30 articles)
- **Part 2** - Job category classification and requirements extraction on the job postings dataset (first 25 postings)

Both parts use LangChain with an LLM served through Groq.

## Setup

Imports, loading the Groq key from the `.env` file and creating the chat model. I am using `openai/gpt-oss-120b` on Groq with temperature 0 so the outputs stay consistent when the notebook is re-run. `reasoning_effort` is set to low because with the default setting the model was spending up to ~2000 thinking tokens on a simple entity extraction, which made it slow, ate into the tokens-per-minute limit and sometimes cut off the JSON answer.

In [1]:
import os
import json
import time
import pandas as pd
from dotenv import load_dotenv

from langchain.chat_models import init_chat_model
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser, JsonOutputParser

pd.set_option("display.max_colwidth", 120)

In [2]:
load_dotenv()
print("Groq key loaded:", os.getenv("GROQ_API_KEY") is not None)

Groq key loaded: True


In [3]:
llm = init_chat_model("openai/gpt-oss-120b", model_provider="groq", temperature=0, reasoning_effort="low")

llm.invoke("Say hello in one short sentence.").content

'Hello!'

The Groq free tier allows only 8000 tokens per minute, so when looping over many articles it is easy to hit a rate limit error. This small helper retries the call after waiting a bit instead of crashing the whole loop.

In [4]:
def run_chain(chain, inputs, retries=6, wait=30):
    for attempt in range(retries):
        try:
            return chain.invoke(inputs)
        except Exception as e:
            print(f"  call failed ({type(e).__name__}), waiting {wait}s and retrying...")
            time.sleep(wait)
    raise RuntimeError("chain kept failing after retries")

---
# Part 1: Topic Detection and Summarization of News Articles

## Step 1: Load the Dataset

The BBC file is tab separated (not comma), so `sep="\t"` is needed. It has 2225 articles with the columns `category`, `filename`, `title` and `content`. As asked, I keep only the first 30 articles.

In [5]:
news_full = pd.read_csv("bbc-news-data.csv", sep="\t")
print("Full dataset shape:", news_full.shape)
news_full.head()

Full dataset shape: (2225, 4)


,category,filename,title,content
0,business,001.txt,Ad sales boost Time Warner profit,"Quarterly profits at US media giant TimeWarner jumped 76% to $1.13bn (£600m) for the three months to December, from..."
1,business,002.txt,Dollar gains on Greenspan speech,The dollar has hit its highest level against the euro in almost three months after the Federal Reserve head said th...
2,business,003.txt,Yukos unit buyer faces loan claim,The owners of embattled Russian oil giant Yukos are to ask the buyer of its former production unit to pay back a $9...
3,business,004.txt,High fuel prices hit BA's profits,British Airways has blamed high fuel prices for a 40% drop in profits. Reporting its results for the three months ...
4,business,005.txt,Pernod takeover talk lifts Domecq,Shares in UK drinks and food firm Allied Domecq have risen on speculation that it could be the target of a takeover...


In [6]:
news_full["category"].value_counts()

category
sport            511
business         510
politics         417
tech             401
entertainment    386
Name: count, dtype: int64

In [7]:
news_df = news_full.head(30).copy()
news_df.insert(0, "Article_ID", range(1, len(news_df) + 1))
print("Working with", len(news_df), "articles")
news_df.head()

Working with 30 articles


,Article_ID,category,filename,title,content
0,1,business,001.txt,Ad sales boost Time Warner profit,"Quarterly profits at US media giant TimeWarner jumped 76% to $1.13bn (£600m) for the three months to December, from..."
1,2,business,002.txt,Dollar gains on Greenspan speech,The dollar has hit its highest level against the euro in almost three months after the Federal Reserve head said th...
2,3,business,003.txt,Yukos unit buyer faces loan claim,The owners of embattled Russian oil giant Yukos are to ask the buyer of its former production unit to pay back a $9...
3,4,business,004.txt,High fuel prices hit BA's profits,British Airways has blamed high fuel prices for a 40% drop in profits. Reporting its results for the three months ...
4,5,business,005.txt,Pernod takeover talk lifts Domecq,Shares in UK drinks and food firm Allied Domecq have risen on speculation that it could be the target of a takeover...


In [8]:
# the first 30 rows are all business articles since the file is sorted by category
news_df["category"].value_counts()

category
business    30
Name: count, dtype: int64

## Step 2: Topic Classification

Prompt template that asks the model to pick exactly one of the five categories. I added three few-shot examples taken from later rows of the dataset (one sport, one tech, one entertainment) so the model sees what each category looks like. At first I had the examples directly above the article and the model was mixing them up with the article it had to classify, so I added a "Now classify the article below" line in between and that fixed it. The chain is `prompt | llm | StrOutputParser` and the output is cleaned with `.strip()` so it is just the label.

In [9]:
# few-shot examples picked from outside the first 30 rows (short snippets only)
example_rows = [
    news_full[news_full["category"] == "sport"].iloc[0],
    news_full[news_full["category"] == "tech"].iloc[0],
    news_full[news_full["category"] == "entertainment"].iloc[0],
]

few_shot_text = ""
for row in example_rows:
    few_shot_text += f"Example article: {row['content'][:300].strip()}...\nCategory: {row['category'].capitalize()}\n\n"

print(few_shot_text)

Example article: British hurdler Sarah Claxton is confident she can win her first major medal at next month's European Indoor Championships in Madrid.  The 25-year-old has already smashed the British record over 60m hurdles twice this season, setting a new mark of 7.96 seconds to win the AAAs title. "I am quite con...
Category: Sport

Example article: The Kyrgyz Republic, a small, mountainous state of the former Soviet republic, is using invisible ink and ultraviolet readers in the country's elections as part of a drive to prevent multiple voting.  This new technology is causing both worries and guarded optimism among different sectors of the po...
Category: Tech

Example article: A Christmas tree that can receive text messages has been unveiled at London's Tate Britain art gallery.  The spruce has an antenna which can receive Bluetooth texts sent by visitors to the Tate. The messages will be "unwrapped" by sculptor Richard Wentworth, who is responsible for decorating the tr...
Categor

In [10]:
classify_prompt = ChatPromptTemplate.from_template(
    """Analyze the following news article and identify its topic as one of the following categories:
Business, Entertainment, Politics, Sport, or Tech.

Here are a few short examples of each category for reference:

{examples}Now classify the article below. Reply with only the category name and nothing else.

Article: {article}
Category:"""
)

classify_chain = classify_prompt | llm | StrOutputParser()

def classify_article(text):
    out = run_chain(classify_chain, {"examples": few_shot_text, "article": text})
    return out.strip().strip(".").capitalize()

In [11]:
# test on one sample article
sample = news_df.iloc[0]
print("Title:", sample["title"])
print("Actual category:", sample["category"])
print("Predicted topic:", classify_article(sample["content"]))

Title: Ad sales boost Time Warner profit
Actual category: business


Predicted topic: Business


In [12]:
# trying one from a different category as well, to make sure it is not just saying Business every time
sample2 = news_full[news_full["category"] == "politics"].iloc[5]
print("Title:", sample2["title"])
print("Actual category:", sample2["category"])
print("Predicted topic:", classify_article(sample2["content"]))

Title: 'Errors' doomed first Dome sale
Actual category: politics


Predicted topic: Politics


## Step 3: Summarization

A simple prompt asking for a 2-3 sentence summary. I told the model to stick to the facts (who / what / when / where) and not add any opinion.

In [13]:
summary_prompt = ChatPromptTemplate.from_template(
    """Summarize the main points of the following news article in 2-3 sentences.
Cover who, what, when and where as applicable. Only state the facts from the article, do not add any personal commentary.

Article: {article}

Summary:"""
)

summary_chain = summary_prompt | llm | StrOutputParser()

def summarize_article(text):
    return run_chain(summary_chain, {"article": text}).strip()

In [14]:
print("Title:", sample["title"])
print()
print(summarize_article(sample["content"]))

Title: Ad sales boost Time Warner profit



Time Warner reported fourth‑quarter profit of $1.13 billion, a 76 % increase from the same period a year earlier, on sales of $11.1 billion, driven by higher high‑speed internet connections and advertising revenue, while its film division saw profits fall 27 % to $284 million. The company now holds an 8 % stake in Google, plans to offer AOL free to its internet customers to boost subscriptions, and must restate its 2000 and 2003 results as the SEC concludes a probe into AOL, including a $300 million settlement offer. For the full year, Time Warner posted a profit of $3.36 billion, up 27 % on 2003, with revenue of $42.09 billion and projected operating‑earnings growth of about 5 % for 2005.


## Step 4: Key Entity Extraction

For entities I ask the model to return JSON with three lists - people, organizations and locations - and parse it with `JsonOutputParser`. This way it is easy to work with the result in Python later. For the dataframe column I flatten the three lists into one `Key_Entities` list, but the grouped version is also shown below for the sample.

In [15]:
entity_prompt = ChatPromptTemplate.from_template(
    """From the article below, list the names of any important people, organizations, or places mentioned.

Return the answer as JSON only, in exactly this format:
{{"people": [...], "organizations": [...], "locations": [...]}}

If there is nothing for a group, return an empty list for it.

Article: {article}"""
)

entity_chain = entity_prompt | llm | JsonOutputParser()

def extract_entities(text):
    result = run_chain(entity_chain, {"article": text})
    # make sure all three keys exist even if the model skipped one
    return {k: result.get(k, []) for k in ["people", "organizations", "locations"]}

In [16]:
print("Title:", sample["title"])
ents = extract_entities(sample["content"])
print(json.dumps(ents, indent=2))

Title: Ad sales boost Time Warner profit


{
  "people": [
    "Richard Parsons"
  ],
  "organizations": [
    "TimeWarner",
    "Google",
    "AOL",
    "Warner Bros",
    "US Securities Exchange Commission (SEC)",
    "Bertelsmann"
  ],
  "locations": [
    "United States"
  ]
}


## Step 5: Update the DataFrame with Results

Now I loop over all 30 articles and run the three chains on each one. The results are collected in lists and then added as new columns `Detected_Topic`, `Summary` and `Key_Entities`. Each article costs roughly 2,800 tokens across the three calls (the article text is sent three times), so with the 8,000 tokens per minute limit I need about 20 seconds per article. The `time.sleep(20)` at the end of each iteration keeps the loop under the limit - without it the loop was failing around article 14 every time.

In [17]:
topics, summaries, entities = [], [], []

start = time.time()
for i, row in news_df.iterrows():
    text = row["content"]
    print(f"[{row['Article_ID']:>2}/30] {row['title'][:60]}")

    topics.append(classify_article(text))
    summaries.append(summarize_article(text))
    ents = extract_entities(text)
    entities.append(ents["people"] + ents["organizations"] + ents["locations"])

    time.sleep(20)

print(f"\nDone in {(time.time() - start)/60:.1f} minutes")

[ 1/30] Ad sales boost Time Warner profit


[ 2/30] Dollar gains on Greenspan speech


[ 3/30] Yukos unit buyer faces loan claim


[ 4/30] High fuel prices hit BA's profits


[ 5/30] Pernod takeover talk lifts Domecq


...

[27/30] Steel firm 'to cut' 45,000 jobs


[28/30] Strong demand triggers oil rally


[29/30] UK firm faces Venezuelan land row


[30/30] Soaring oil 'hits world economy'



Done in 11.1 minutes


In [18]:
news_df["Detected_Topic"] = topics
news_df["Summary"] = summaries
news_df["Key_Entities"] = entities

news_df[["Article_ID", "title", "Detected_Topic", "Summary", "Key_Entities"]].head(10)

,Article_ID,title,Detected_Topic,Summary,Key_Entities
0,1,Ad sales boost Time Warner profit,Business,"Time Warner reported fourth‑quarter profit of $1.13 billion for the three months ended December, a 76 % increase fro...","[Richard Parsons, TimeWarner, Google, AOL, Warner Bros, US Securities Exchange Commission (SEC), Bertelsmann, United..."
1,2,Dollar gains on Greenspan speech,Business,"The dollar rose to $1.2871 per euro, its highest level in almost three months, after Federal Reserve Chairman Alan G...","[Alan Greenspan, Robert Sinche, Federal Reserve, Bank of America, G7, White House, New York, London, China, Beijing]"
2,3,Yukos unit buyer faces loan claim,Business,"Menatep Group, the owner of the former Yukos production unit Yugansk, will ask Rosneft— which bought Yugansk for $9....","[Jamie Firestone, Tim Osborne, Mikhail Khodorkovsky, Yukos, Rosneft, Menatep Group, Yugansk, Reuters, Russia, Moscow..."
3,4,High fuel prices hit BA's profits,Business,"British Airways reported a 40 % drop in pre‑tax profit for the three months to 31 December 2004, posting £75 million...","[Rod Eddington, Mike Powell, Martin Broughton, Nick Van den Brul, British Airways, Dresdner Kleinwort Wasserstein, B..."
4,5,Pernod takeover talk lifts Domecq,Business,Allied Domecq’s London shares rose about 4% after Wall Street Journal and Financial Times reports that France’s Pern...,"[Allied Domecq, Pernod Ricard, Wall Street Journal, Financial Times, Seagram, Diageo, Glenmorangie, LVMH, Chivas Reg..."
5,6,Japan narrowly escapes recession,Business,"Japan’s economy barely grew 0.1% in the three months to September 2023, following a similar‑sized contraction in the...","[Heizo Takenaka, Paul Sheard, Lehman Brothers, Japan, Tokyo]"
6,7,Jobs growth still slow in the US,Business,"In January, U.S. firms added 146,000 non‑farm jobs, falling short of the Labor Department’s forecast of 190,000, whi...","[President Bush, Herbert Hoover, Rick Egelton, Ken Mayland, Labor Department, BMO Financial Group, ClearView Economi..."
7,8,India calls for fair trade rules,Politics,India’s finance minister Palaniappan Chidambaram attended the G7 summit in London on 23‑24 May 2005 as part of the G...,"[Palaniappan Chidambaram, Gordon Brown, G7, United Nations, World Bank, IMF, G20, London, India, China, Brazil, Sout..."
8,9,Ethiopia's crop production up 24%,Business,"In 2004 Ethiopia produced 14.27 million tonnes of crops, a 24 % increase over 2003’s 11.49 million tonnes and 21 % a...","[Henri Josserand, Food and Agriculture Organisation, World Food Programme, FAO's Global Information and Early Warnin..."
9,10,Court rejects $280bn tobacco case,Business,A U.S. Court of Appeals for the District of Columbia rejected a 1999 Clinton‑administration lawsuit that sought $280...,"[US government, Clinton administration, Altria Group, RJ Reynolds Tobacco, Lorillard Tobacco, Liggett Group, Brown a..."


In [19]:
# quick check - how many predicted topics match the original category label
news_df["correct"] = news_df["Detected_Topic"].str.lower() == news_df["category"].str.lower()
print("Accuracy on the 30 articles:", news_df["correct"].mean())
news_df["Detected_Topic"].value_counts()

Accuracy on the 30 articles: 0.9666666666666667


Detected_Topic
Business    29
Politics     1
Name: count, dtype: int64

In [20]:
# final dataframe with all original and new columns together
news_final = news_df.drop(columns=["correct"])
print(news_final.columns.tolist())
news_final

['Article_ID', 'category', 'filename', 'title', 'content', 'Detected_Topic', 'Summary', 'Key_Entities']


,Article_ID,category,filename,title,content,Detected_Topic,Summary,Key_Entities
0,1,business,001.txt,Ad sales boost Time Warner profit,"Quarterly profits at US media giant TimeWarner jumped 76% to $1.13bn (£600m) for the three months to December, from...",Business,"Time Warner reported fourth‑quarter profit of $1.13 billion for the three months ended December, a 76 % increase fro...","[Richard Parsons, TimeWarner, Google, AOL, Warner Bros, US Securities Exchange Commission (SEC), Bertelsmann, United..."
1,2,business,002.txt,Dollar gains on Greenspan speech,The dollar has hit its highest level against the euro in almost three months after the Federal Reserve head said th...,Business,"The dollar rose to $1.2871 per euro, its highest level in almost three months, after Federal Reserve Chairman Alan G...","[Alan Greenspan, Robert Sinche, Federal Reserve, Bank of America, G7, White House, New York, London, China, Beijing]"
2,3,business,003.txt,Yukos unit buyer faces loan claim,The owners of embattled Russian oil giant Yukos are to ask the buyer of its former production unit to pay back a $9...,Business,"Menatep Group, the owner of the former Yukos production unit Yugansk, will ask Rosneft— which bought Yugansk for $9....","[Jamie Firestone, Tim Osborne, Mikhail Khodorkovsky, Yukos, Rosneft, Menatep Group, Yugansk, Reuters, Russia, Moscow..."
3,4,business,004.txt,High fuel prices hit BA's profits,British Airways has blamed high fuel prices for a 40% drop in profits. Reporting its results for the three months ...,Business,"British Airways reported a 40 % drop in pre‑tax profit for the three months to 31 December 2004, posting £75 million...","[Rod Eddington, Mike Powell, Martin Broughton, Nick Van den Brul, British Airways, Dresdner Kleinwort Wasserstein, B..."
4,5,business,005.txt,Pernod takeover talk lifts Domecq,Shares in UK drinks and food firm Allied Domecq have risen on speculation that it could be the target of a takeover...,Business,Allied Domecq’s London shares rose about 4% after Wall Street Journal and Financial Times reports that France’s Pern...,"[Allied Domecq, Pernod Ricard, Wall Street Journal, Financial Times, Seagram, Diageo, Glenmorangie, LVMH, Chivas Reg..."
5,6,business,006.txt,Japan narrowly escapes recession,"Japan's economy teetered on the brink of a technical recession in the three months to September, figures show. Rev...",Business,"Japan’s economy barely grew 0.1% in the three months to September 2023, following a similar‑sized contraction in the...","[Heizo Takenaka, Paul Sheard, Lehman Brothers, Japan, Tokyo]"
6,7,business,007.txt,Jobs growth still slow in the US,"The US created fewer jobs than expected in January, but a fall in jobseekers pushed the unemployment rate to its lo...",Business,"In January, U.S. firms added 146,000 non‑farm jobs, falling short of the Labor Department’s forecast of 190,000, whi...","[President Bush, Herbert Hoover, Rick Egelton, Ken Mayland, Labor Department, BMO Financial Group, ClearView Economi..."
7,8,business,008.txt,India calls for fair trade rules,"India, which attends the G7 meeting of seven leading industrialised nations on Friday, is unlikely to be cowed by i...",Politics,India’s finance minister Palaniappan Chidambaram attended the G7 summit in London on 23‑24 May 2005 as part of the G...,"[Palaniappan Chidambaram, Gordon Brown, G7, United Nations, World Bank, IMF, G20, London, India, China, Brazil, Sout..."
8,9,business,009.txt,Ethiopia's crop production up 24%,"Ethiopia produced 14.27 million tonnes of crops in 2004, 24% higher than in 2003 and 21% more than the average of t...",Business,"In 2004 Ethiopia produced 14.27 million tonnes of crops, a 24 % increase over 2003’s 11.49 million tonnes and 21 % a...","[Henri Josserand, Food and Agriculture Organisation, World Food Programme, FAO's Global Information and Early Warnin..."
9,10,business,010.txt,Court rejects $280bn tobacco case,A US government claim accusing the country's biggest tobacco companies of covering 

In [21]:
# one record in the JSON layout shown in the assignment
record = news_final.iloc[0]
print(json.dumps({
    "Article_ID": int(record["Article_ID"]),
    "Title": record["title"],
    "Article_Text": record["content"][:150] + "... [excerpt]",
    "Detected_Topic": record["Detected_Topic"],
    "Summary": record["Summary"],
    "Key_Entities": record["Key_Entities"],
}, indent=2))

{
  "Article_ID": 1,
  "Title": "Ad sales boost Time Warner profit",
  "Article_Text": " Quarterly profits at US media giant TimeWarner jumped 76% to $1.13bn (\u00a3600m) for the three months to December, from $639m year-earlier.  The firm, wh... [excerpt]",
  "Detected_Topic": "Business",
  "Summary": "Time Warner reported fourth\u2011quarter profit of $1.13\u202fbillion for the three months ended December, a 76\u202f% increase from the same period a year earlier, with sales rising 2\u202f% to $11.1\u202fbillion; the gains were driven by higher high\u2011speed internet sales, stronger advertising revenue and one\u2011off items that offset a dip at Warner\u202fBros. The company, which now holds an 8\u202f% stake in Google, said its AOL division lost 464,000 subscribers but saw underlying profit before exceptional items rise 8\u202f% on better internet ad sales, and it plans to offer AOL free to Time Warner internet customers. For the full year, Time Warner posted a profit of $3.36\u202

In [22]:
news_final.to_csv("part1_news_results.csv", index=False)
print("saved part1_news_results.csv")

saved part1_news_results.csv


---
# Part 2: Job Postings Analysis - Role Categorization and Requirements Extraction

## Step 1: Load the Dataset

The job postings CSV has an unnamed index column plus `Job Title` and `Job Description`. I drop the index column, rename the other two to `Job_Title` and `Job_Description` and keep the first 25 postings.

In [23]:
jobs_full = pd.read_csv("job_title_des.csv")
jobs_full = jobs_full.drop(columns=["Unnamed: 0"])
jobs_full = jobs_full.rename(columns={"Job Title": "Job_Title", "Job Description": "Job_Description"})
print("Full dataset shape:", jobs_full.shape)
jobs_full.head()

Full dataset shape: (2277, 2)


,Job_Title,Job_Description
0,Flutter Developer,We are looking for hire experts flutter developer. So you are eligible this post then apply your resume.\nJob Types:...
1,Django Developer,PYTHON/DJANGO (Developer/Lead) - Job Code(PDJ - 04)\nStrong Python experience in API development (REST/RPC).\nExperi...
2,Machine Learning,"Data Scientist (Contractor)\n\nBangalore, IN\n\nResponsibilities\n\nWe are looking for a capable data scientist to j..."
3,iOS Developer,JOB DESCRIPTION:\n\nStrong framework outside of iOS is always a plus\n\niOS experience and generalist engineers with...
4,Full Stack Developer,job responsibility full stack engineer – react role make impact petsmart transforming engineering team meet need rap...


In [24]:
jobs_df = jobs_full.head(25).copy().reset_index(drop=True)
print("Working with", len(jobs_df), "job postings")
jobs_df["Job_Title"].value_counts()

Working with 25 job postings


Job_Title
Database Administrator    4
Machine Learning          3
Software Engineer         3
iOS Developer             2
Full Stack Developer      2
Java Developer            2
JavaScript Developer      2
DevOps Engineer           2
Wordpress Developer       2
Flutter Developer         1
Django Developer          1
PHP Developer             1
Name: count, dtype: int64

In [25]:
# looking at one full description to understand the format
print(jobs_df.loc[1, "Job_Title"])
print("-" * 60)
print(jobs_df.loc[1, "Job_Description"])

Django Developer
------------------------------------------------------------
PYTHON/DJANGO (Developer/Lead) - Job Code(PDJ - 04)
Strong Python experience in API development (REST/RPC).
Experience working with API Frameworks (Django/flask).
Experience evaluating and improving the efficiency of programs in a Linux environment.
Ability to effectively handle multiple tasks with a high level of accuracy and attention to detail.
Good verbal and written communication skills.
Working knowledge of SQL.
JSON experience preferred.
Good knowledge in automated unit testing using PyUnit.


## Step 2: Job Category Classification

The prompt gets both the title and the description and has to choose one domain from a fixed list. I extended the list from the assignment a bit (Sales, Human Resources, Design, Operations) with `Other` as fallback. Two small examples are included in the prompt to show the expected format. Descriptions are cut to 2500 characters since some postings are very long and the important part is usually at the top. In my first attempt the whole posting was squeezed into one line (`Job: ... Description: ...`) and for a few rows the model replied with things like "Please provide the job title" or picked Other for obvious developer roles, so I split the title and description onto separate lines and added a "Now categorize this job posting" line before them. I also map anything that is not in the list to `Other` just to be safe.

In [26]:
job_categories = ["Technology/IT", "Finance", "Marketing", "Healthcare", "Education",
                  "Sales", "Human Resources", "Design", "Operations", "Other"]

category_prompt = ChatPromptTemplate.from_template(
    """You are classifying job postings into broad domains.
Categorize the job posting into one of the following domains: {categories}

Use "Other" only if the job clearly does not fit any of the listed domains.
Reply with only the domain name, nothing else.

Examples:
Job title: Digital Marketing Executive
Description: Manage social media campaigns, SEO and email marketing for our brand.
Domain: Marketing

Job title: Staff Nurse
Description: Provide patient care in the ICU, administer medication and maintain patient records.
Domain: Healthcare

Now categorize this job posting.
Job title: {title}
Description: {description}
Domain:"""
)

category_chain = category_prompt | llm | StrOutputParser()

def classify_job(title, description):
    out = run_chain(category_chain, {
        "categories": ", ".join(job_categories),
        "title": title,
        "description": description[:2500],
    })
    out = out.strip().strip(".")
    # fallback in case the model replies with something outside the list
    return out if out in job_categories else "Other"

In [27]:
job_sample = jobs_df.iloc[1]
print("Job title:", job_sample["Job_Title"])
print("Predicted category:", classify_job(job_sample["Job_Title"], job_sample["Job_Description"]))

Job title: Django Developer


Predicted category: Technology/IT


In [28]:
# checking with a made up non-tech posting to see that other categories also come out
print(classify_job("Accounts Payable Clerk",
                   "Process vendor invoices, reconcile bank statements and support monthly closing. Tally and Excel required."))

Finance


## Step 3: Requirements Extraction

I went with a single composite prompt that returns JSON with three fields - `Skills` (a list), `Education` and `Experience`. If something is not mentioned in the description the model is told to write `"Not specified"`, and I also handle that in Python in case it returns null or an empty value.

In [29]:
extract_prompt = ChatPromptTemplate.from_template(
    """Extract the required skills, education level, and years of experience from the job description below.

Return JSON only, in exactly this format:
{{
  "Skills": ["skill 1", "skill 2", ...],
  "Education": "minimum education level required or preferred",
  "Experience": "minimum years of experience or experience level required"
}}

Rules:
- Skills should be specific skills, technologies, tools or domain knowledge (for example Python, SQL, AWS, project management).
- If education is not mentioned, put "Not specified".
- If experience is not mentioned, put "Not specified".
- Keep Education and Experience short (a few words).

Job description:
{description}"""
)

extract_chain = extract_prompt | llm | JsonOutputParser()

def clean_value(v):
    # null / empty / none-ish answers all become "Not specified"
    if v is None or str(v).strip() == "" or str(v).strip().lower() in ["none", "null", "n/a", "not mentioned"]:
        return "Not specified"
    return str(v).strip()

def extract_requirements(description):
    result = run_chain(extract_chain, {"description": description[:4000]})
    skills = result.get("Skills") or []
    if isinstance(skills, str):
        skills = [s.strip() for s in skills.split(",") if s.strip()]
    return {
        "Skills": skills if skills else ["Not specified"],
        "Education": clean_value(result.get("Education")),
        "Experience": clean_value(result.get("Experience")),
    }

In [30]:
print("Job title:", job_sample["Job_Title"])
print(json.dumps(extract_requirements(job_sample["Job_Description"]), indent=2))

Job title: Django Developer


{
  "Skills": [
    "Python",
    "API development (REST/RPC)",
    "Django",
    "Flask",
    "Linux",
    "SQL",
    "JSON",
    "Automated unit testing (PyUnit)",
    "Verbal and written communication"
  ],
  "Education": "Not specified",
  "Experience": "Not specified"
}


In [31]:
# the first posting (Flutter Developer) barely has any requirements written, good test for "Not specified"
print(jobs_df.loc[0, "Job_Title"])
print(json.dumps(extract_requirements(jobs_df.loc[0, "Job_Description"]), indent=2))

Flutter Developer


{
  "Skills": [
    "Flutter"
  ],
  "Education": "Not specified",
  "Experience": "1 year (Preferred)"
}


## Step 4: Apply the LLM Chain to Each Job Posting

Loop through the 25 postings, running the classification prompt and then the extraction prompt for each one. Results go into lists that become the new columns in the next step. Same 20 second pause per posting as in Part 1, since the descriptions are sent twice and some of them are long.

In [32]:
categories, skills_list, education_list, experience_list = [], [], [], []

start = time.time()
for i, row in jobs_df.iterrows():
    print(f"[{i+1:>2}/25] {row['Job_Title']}")

    categories.append(classify_job(row["Job_Title"], row["Job_Description"]))

    req = extract_requirements(row["Job_Description"])
    skills_list.append(req["Skills"])
    education_list.append(req["Education"])
    experience_list.append(req["Experience"])

    time.sleep(20)

print(f"\nDone in {(time.time() - start)/60:.1f} minutes")

[ 1/25] Flutter Developer


[ 2/25] Django Developer


[ 3/25] Machine Learning


[ 4/25] iOS Developer


[ 5/25] Full Stack Developer


...

[22/25] Wordpress Developer


[23/25] JavaScript Developer


[24/25] Database Administrator


[25/25] PHP Developer



Done in 8.9 minutes


## Step 5: Update the DataFrame with New Columns

Adding `Predicted_Category`, `Required_Skills`, `Education_Required` and `Experience_Required`. `Required_Skills` is kept as a Python list for every row so the column is consistent.

In [33]:
jobs_df["Predicted_Category"] = categories
jobs_df["Required_Skills"] = skills_list
jobs_df["Education_Required"] = education_list
jobs_df["Experience_Required"] = experience_list

jobs_df[["Job_Title", "Predicted_Category", "Required_Skills", "Education_Required", "Experience_Required"]]

,Job_Title,Predicted_Category,Required_Skills,Education_Required,Experience_Required
0,Flutter Developer,Technology/IT,[Flutter],Not specified,1 year (Preferred)
1,Django Developer,Technology/IT,"[Python, API development (REST/RPC), Django, Flask, Linux, SQL, JSON, Automated unit testing (PyUnit), Verbal and wr...",Not specified,Not specified
2,Machine Learning,Technology/IT,"[Python, Java, Machine Learning, Deep Learning, PyTorch, TensorFlow, Keras, Spark, Big Data, Statistics, Applied Mat...","Graduate or M.Sc. in Computer Science, Mathematics or equivalent",At least 3 years
3,iOS Developer,Technology/IT,"[iOS development, Objective-C, Cocoa Touch, Core Data, Core Animation, Core Graphics, Core Text, Networking, Concurr...",Not specified,Not specified
4,Full Stack Developer,Technology/IT,"[React, React Native, JavaScript, HTML, CSS, RESTful APIs, HTTP, Redux, Angular, Vue, MVC, Object‑oriented programmi...",Not specified,5+ years
5,Java Developer,Technology/IT,"[C#, .NET, .NET Core, HTML5, CSS3, MsSQL, MySQL, ReactJS, Web services (WSDL, SOAP, RESTful), Relational databases, ...","Bachelor's Degree in Computer Science, Information Systems, or related field",2 years of software development
6,Full Stack Developer,Technology/IT,"[Node.js, Java, NoSQL, MongoDB, Elasticsearch, Redis, React, Angular, JavaScript, HTML, CSS, SPA development, Respon...",B.Sc in Computer Science or Engineering (or equivalent),Minimum 2 years
7,JavaScript Developer,Technology/IT,"[ReactJS, NodeJS, Azure Functions, GraphQL, HTML5, CSS3, JavaScript, REST]",Any graduation,3 years
8,DevOps Engineer,Technology/IT,"[Bash, Ruby, Python, Java, Puppet, Chef, Cloudify, CFEngine, Cobbler, Foreman, PHP, Linux, Windows, Application debu...",Not specified,Not specified
9,Software Engineer,Technology/IT,"[REST API, C/C++, Linux/Unix, Python, Go, Git, Gerrit, Jenkins, cloud deployment, VMs, containers, control plane API...",BS or MS in Computer Engineering/Science,Minimum 7 years software development


In [34]:
print(jobs_df["Predicted_Category"].value_counts())
print()
print("Education not specified:", (jobs_df["Education_Required"] == "Not specified").sum(), "of 25")
print("Experience not specified:", (jobs_df["Experience_Required"] == "Not specified").sum(), "of 25")

Predicted_Category
Technology/IT    25
Name: count, dtype: int64

Education not specified: 9 of 25
Experience not specified: 6 of 25


### Spot check

Comparing the extracted fields with the actual description text for a couple of postings to verify the outputs make sense.

In [35]:
for idx in [1, 8]:
    row = jobs_df.loc[idx]
    print("=" * 80)
    print("Job title:", row["Job_Title"], "|", row["Predicted_Category"])
    print("Skills:", row["Required_Skills"])
    print("Education:", row["Education_Required"])
    print("Experience:", row["Experience_Required"])
    print("-" * 80)
    print(row["Job_Description"][:900])
    print()

Job title: Django Developer | Technology/IT
Skills: ['Python', 'API development (REST/RPC)', 'Django', 'Flask', 'Linux', 'SQL', 'JSON', 'Automated unit testing (PyUnit)', 'Verbal and written communication']
Education: Not specified
Experience: Not specified
--------------------------------------------------------------------------------
PYTHON/DJANGO (Developer/Lead) - Job Code(PDJ - 04)
Strong Python experience in API development (REST/RPC).
Experience working with API Frameworks (Django/flask).
Experience evaluating and improving the efficiency of programs in a Linux environment.
Ability to effectively handle multiple tasks with a high level of accuracy and attention to detail.
Good verbal and written communication skills.
Working knowledge of SQL.
JSON experience preferred.
Good knowledge in automated unit testing using PyUnit.

Job title: DevOps Engineer | Technology/IT
Skills: ['Bash', 'Ruby', 'Python', 'Java', 'Puppet', 'Chef', 'Cloudify', 'CFEngine', 'Cobbler', 'Foreman', 'PHP',

In [36]:
# final dataframe with all original and new columns together
print(jobs_df.columns.tolist())
jobs_df

['Job_Title', 'Job_Description', 'Predicted_Category', 'Required_Skills', 'Education_Required', 'Experience_Required']


,Job_Title,Job_Description,Predicted_Category,Required_Skills,Education_Required,Experience_Required
0,Flutter Developer,We are looking for hire experts flutter developer. So you are eligible this post then apply your resume.\nJob Types:...,Technology/IT,[Flutter],Not specified,1 year (Preferred)
1,Django Developer,PYTHON/DJANGO (Developer/Lead) - Job Code(PDJ - 04)\nStrong Python experience in API development (REST/RPC).\nExperi...,Technology/IT,"[Python, API development (REST/RPC), Django, Flask, Linux, SQL, JSON, Automated unit testing (PyUnit), Verbal and wr...",Not specified,Not specified
2,Machine Learning,"Data Scientist (Contractor)\n\nBangalore, IN\n\nResponsibilities\n\nWe are looking for a capable data scientist to j...",Technology/IT,"[Python, Java, Machine Learning, Deep Learning, PyTorch, TensorFlow, Keras, Spark, Big Data, Statistics, Applied Mat...","Graduate or M.Sc. in Computer Science, Mathematics or equivalent",At least 3 years
3,iOS Developer,JOB DESCRIPTION:\n\nStrong framework outside of iOS is always a plus\n\niOS experience and generalist engineers with...,Technology/IT,"[iOS development, Objective-C, Cocoa Touch, Core Data, Core Animation, Core Graphics, Core Text, Networking, Concurr...",Not specified,Not specified
4,Full Stack Developer,job responsibility full stack engineer – react role make impact petsmart transforming engineering team meet need rap...,Technology/IT,"[React, React Native, JavaScript, HTML, CSS, RESTful APIs, HTTP, Redux, Angular, Vue, MVC, Object‑oriented programmi...",Not specified,5+ years
5,Java Developer,Software Developer - Integration*\nImmediate Opening!*\nA dynamic Akron / Cleveland area company is looking for an e...,Technology/IT,"[C#, .NET, .NET Core, HTML5, CSS3, MsSQL, MySQL, ReactJS, Web services (WSDL, SOAP, RESTful), Relational databases, ...","Bachelor's Degree in Computer Science, Information Systems, or related field",2 years of software development
6,Full Stack Developer,senior full stack developer \- 1800026h cwt looking senior full stack developer proven back-end skill well strong fr...,Technology/IT,"[Node.js, Java, NoSQL, MongoDB, Elasticsearch, Redis, React, Angular, JavaScript, HTML, CSS, SPA development, Respon...",B.Sc in Computer Science or Engineering (or equivalent),Minimum 2 years
7,JavaScript Developer,"Job Description:\n\nReactJS + NodeJs, Azure Functions, and GraphQL capability\n\nStrong hands on experience on javas...",Technology/IT,"[ReactJS, NodeJS, Azure Functions, GraphQL, HTML5, CSS3, JavaScript, REST]",Any graduation,3 years
8,DevOps Engineer,"Main Responsibilities and Deliverables:\nManage/support the rollout, scalability and execution of the automation as ...",Technology/IT,"[Bash, Ruby, Python, Java, Puppet, Chef, Cloudify, CFEngine, Cobbler, Foreman, PHP, Linux, Windows, Application debu...",Not specified,Not specified
9,Software Engineer,"Overview\n\n\nBased in Silicon Valley, Tintri is a wholly owned subsidiary of DataDirect Networks (DDN.com), the dat...",Technology/IT,"[REST API, C/C++, Linux/Unix, Python, Go, Git, Gerrit, Jenkins, cloud deployment, VMs, containers, control plane API...",BS or MS in Computer Engineering/Science,Minimum 7 years software development


In [37]:
# one record in the JSON layout shown in the assignment
record = jobs_df.iloc[1]
print(json.dumps({
    "Job_Title": record["Job_Title"],
    "Job_Description": record["Job_Description"][:200] + "...",
    "Predicted_Category": record["Predicted_Category"],
    "Required_Skills": record["Required_Skills"],
    "Education_Required": record["Education_Required"],
    "Experience_Required": record["Experience_Required"],
}, indent=2))

{
  "Job_Title": "Django Developer",
  "Job_Description": "PYTHON/DJANGO (Developer/Lead) - Job Code(PDJ - 04)\nStrong Python experience in API development (REST/RPC).\nExperience working with API Frameworks (Django/flask).\nExperience evaluating and improving t...",
  "Predicted_Category": "Technology/IT",
  "Required_Skills": [
    "Python",
    "API development (REST/RPC)",
    "Django",
    "Flask",
    "Linux",
    "SQL",
    "JSON",
    "Automated unit testing (PyUnit)",
    "Verbal and written communication"
  ],
  "Education_Required": "Not specified",
  "Experience_Required": "Not specified"
}


In [38]:
jobs_df.to_csv("part2_jobs_results.csv", index=False)
print("saved part2_jobs_results.csv")

saved part2_jobs_results.csv


---
## Note on the bonus (full dataset)

I did not attempt the bonus. Running Part 1 on all 2225 articles would be about 6,700 LLM calls and Part 2 on all 2277 postings another 4,500, which is well over the 1000 requests per day limit on the Groq free tier. I also tried `llama3.2:1b` through Ollama as suggested, but on my laptop (no dedicated GPU support) a single article took around 30-40 seconds per call, so the full run would have taken more than a day. The pipeline itself works the same way for any number of rows, only the `head(30)` / `head(25)` would need to be removed.